# Binary field addition

Adapted from Qualtran v0.7.0 at bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2024 Google LLC, Apache-2.0. Native metadata, Compiler declarations and Drawer tables retain all original example instances and parameter/register topics. The field type is resolved through Simulation. The native signature table layout and resource labels differ from the original display.


In [ ]:
from IPython.display import SVG, display
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from examples.algorithms.field_addition_catalog import field_addition_examples, field_signature_svg

## Register field addition

For a degree-m binary field, add x into y using bitwise XOR. The normal operation is |x>|y> -> |x>|x XOR y>.

`binary_field_encoding` resolves an existing native field encoding, a degree m, a polynomial, or its descending nonzero powers. `gf2_add_ir(source, target)` receives two disjoint equal-width big-endian words. The x and y words are THRU registers, x is preserved, and there is no additional workspace. `field_addition_calls(m)` declares exactly m CNOT calls without constructing the circuit. Symbolic declarations are supported by the model while concrete wire tuples are required for execution.


In [ ]:
from flagquantum.compiler.registers.finite_field.coefficients import gf2_add_ir, gf2_add_constant_ir
from examples.algorithms.binary_field_arithmetic import FieldOperation
import flagquantum as fq
import torch

### Example Instances

In [ ]:
examples = field_addition_examples(constant=False)
concrete = examples[0]
concrete

In [ ]:
symbolic = examples[1]
symbolic

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg(examples)))

### Call Graph

In [ ]:
name, signature, calls = concrete
root = name
# The existing graph owner accepts declared calls without allocating a circuit.
graph = symbolic_call_graph(root, {root: calls}, max_depth=1)
display(SVG(render_call_graph_svg(export_call_graph_dot(root, graph.direct_calls))))
print("Declared primitive leaves", graph.leaf_counts)

## Execute through append

Words are big-endian. Addition preserves the source, updates the target with XOR and is its own inverse. There are zero work wires and zero measurements, so there is a single coherent execution branch. Actual emitted primitives are listed separately from the symbolic source model. The symbolic constant count is an upper bound of one X per bit.

In [ ]:
state = torch.zeros(256, dtype=torch.complex128)
state[(3 << 4) | 5] = 1
instructions = gf2_add_ir((0, 1, 2, 3), (4, 5, 6, 7))
circuit = fq.Circuit(8, inputs=state, dtype=torch.complex128).append(FieldOperation(8, instructions))
assert int(circuit.state().abs().argmax()) == (3 << 4) | 6
from collections import Counter
print("Actual emitted primitives", dict(Counter(g.name for g in instructions)))
print("Source model", concrete[2])
assert torch.allclose(circuit.append(FieldOperation(8, instructions)).state().flatten(), state)